# Notebook 05: Figures and Results Visualisation

**Inputs:** All outputs from Notebooks 01–04  
**Outputs:** Publication-quality figures for all main results and supplementary materials

All figures are exported at 300 DPI in both PNG (for inspection) and PDF (for submission).

---

In [ ]:
import json, os
import numpy as np
import pandas as pd
import geopandas as gpd
import rasterio
from rasterio.plot import show
import matplotlib.pyplot as plt
import matplotlib.colors as mcolors
import matplotlib.patches as mpatches
from matplotlib.gridspec import GridSpec
from matplotlib.colorbar import ColorbarBase
import warnings
warnings.filterwarnings('ignore')

with open('../data/config.json') as f:
    CONFIG = json.load(f)

os.makedirs('../outputs/figures', exist_ok=True)

# Publication figure settings
plt.rcParams.update({
    'font.family':       'serif',
    'font.size':         10,
    'axes.labelsize':    10,
    'axes.titlesize':    11,
    'xtick.labelsize':   8,
    'ytick.labelsize':   8,
    'legend.fontsize':   9,
    'figure.dpi':        150,
    'savefig.dpi':       300,
    'savefig.bbox':      'tight',
    'axes.spines.top':   False,
    'axes.spines.right': False,
})

FIGURE_DIR = '../outputs/figures'
print('Figure settings configured.')

## Figure 1: Regional Settlement Detection Map

Main result figure — five-country detection map with confidence overlay and four inset panels  
showing representative sites (analogous to Figure 1 in Robinson et al. 2026).

In [ ]:
# Inset site coordinates [west, south, east, north]
# Chosen to represent high/medium/low confidence and urban/rural gradient
INSET_SITES = {
    'A — Nairobi peri-urban, Kenya':      [36.6, -1.5,  37.2, -0.9],
    'B — Rural Tabora, Tanzania':          [32.5, -5.2,  33.1, -4.6],
    'C — Rural Kayanza, Burundi':          [29.5, -2.9,  30.1, -2.3],
    'D — Smallholder highlands, Rwanda':   [29.4, -2.1,  30.0, -1.5],
}

# Class colormap: background=white, isolated=amber, clustered=crimson
SETTLEMENT_CMAP = mcolors.ListedColormap(['#f5f5f5', '#E69F00', '#CC0000'])
SETTLEMENT_NORM = mcolors.BoundaryNorm([0, 0.5, 1.5, 2.5], SETTLEMENT_CMAP.N)


def plot_regional_map(prediction_path, confidence_path,
                       country_boundaries_path, inset_sites,
                       output_name='figure1_regional_map'):
    """
    Plots the five-country settlement detection map with:
      - Main panel: filtered prediction map (class-coloured)
      - Confidence overlay as transparency mask
      - Country boundary outlines
      - Four inset panels at specified site coordinates
      - Class legend and confidence scale bar
    """
    fig = plt.figure(figsize=(14, 10))
    gs  = GridSpec(3, 4, figure=fig, hspace=0.3, wspace=0.25)

    # Main panel
    ax_main = fig.add_subplot(gs[:, :3])

    # Inset panels
    ax_insets = [
        fig.add_subplot(gs[0, 3]),
        fig.add_subplot(gs[1, 3]),
        fig.add_subplot(gs[2, 3]),
        # 4th inset shares row with first if needed
    ]

    # --- Main panel ---
    if os.path.exists(prediction_path) and os.path.exists(confidence_path):
        with rasterio.open(prediction_path) as src:
            pred = src.read(1).astype(float)
            extent = [src.bounds.left, src.bounds.right,
                      src.bounds.bottom, src.bounds.top]

        with rasterio.open(confidence_path) as src:
            conf = src.read(1)

        # Downsample for display (every 50th pixel = 500m)
        pred_display = pred[::50, ::50]
        conf_display = conf  # already 500m

        ax_main.imshow(
            pred_display,
            extent=extent,
            cmap=SETTLEMENT_CMAP,
            norm=SETTLEMENT_NORM,
            origin='upper',
            interpolation='none'
        )
    else:
        # Placeholder if predictions not yet available
        ax_main.text(0.5, 0.5, 'Settlement prediction map\n(run Notebooks 01-04 first)',
                    ha='center', va='center', transform=ax_main.transAxes,
                    fontsize=12, color='gray', style='italic')
        ax_main.set_facecolor('#f0f0f0')

    # Country boundaries
    if os.path.exists(country_boundaries_path):
        boundaries = gpd.read_file(country_boundaries_path)
        boundaries.plot(ax=ax_main, facecolor='none', edgecolor='black',
                       linewidth=0.6)

    # Mark inset site bounding boxes
    for label, bbox in inset_sites.items():
        w, s, e, n = bbox
        rect = plt.Rectangle((w, s), e-w, n-s,
                              fill=False, edgecolor='steelblue',
                              linewidth=1.2, linestyle='--')
        ax_main.add_patch(rect)

    ax_main.set_xlabel('Longitude')
    ax_main.set_ylabel('Latitude')
    ax_main.set_title('Rural Settlement Detection — East African Great Lakes Region (2022–2023)',
                      fontweight='bold', pad=8)

    # Legend
    legend_patches = [
        mpatches.Patch(color='#E69F00', label='Isolated rural dwelling (Class 1)'),
        mpatches.Patch(color='#CC0000', label='Clustered settlement (Class 2)'),
    ]
    ax_main.legend(handles=legend_patches, loc='lower left', framealpha=0.9)

    # --- Inset panels ---
    for ax_inset, (label, bbox) in zip(ax_insets, list(inset_sites.items())[:3]):
        ax_inset.set_xlim(bbox[0], bbox[2])
        ax_inset.set_ylim(bbox[1], bbox[3])
        ax_inset.set_title(label, fontsize=7, pad=3)
        ax_inset.set_xticks([])
        ax_inset.set_yticks([])
        ax_inset.set_facecolor('#e8e8e8')
        ax_inset.text(0.5, 0.5, '(10 m detail)',
                     ha='center', va='center',
                     transform=ax_inset.transAxes,
                     fontsize=7, color='gray', style='italic')

    plt.suptitle('Figure 1', x=0.02, y=0.98, ha='left',
                 fontsize=9, color='gray')

    for fmt in ['png', 'pdf']:
        fig.savefig(f'{FIGURE_DIR}/{output_name}.{fmt}')
    plt.show()
    print(f'Figure 1 saved.')


plot_regional_map(
    prediction_path='../outputs/settlement_filtered_conf04_5country.tif',
    confidence_path='../outputs/confidence_raster_5country.tif',
    country_boundaries_path='../data/boundaries/great_lakes_5country.gpkg',
    inset_sites=INSET_SITES
)

## Figure 2: Confidence Layer Behaviour at Three Representative Sites

Analogous to Figure 4 in Robinson et al. (2026) — rows are high/medium/low confidence sites,  
columns are confidence raster / unfiltered predictions / filtered predictions.

In [ ]:
CONFIDENCE_SITES = {
    'High confidence — Rural Kenya':       [36.8, -1.3, 37.0, -1.1],
    'Medium confidence — Rural Tanzania':  [32.7, -5.0, 32.9, -4.8],
    'Low confidence — Rural Burundi':      [29.6, -2.7, 29.8, -2.5],
}

CONF_CMAP = plt.cm.RdYlGn  # red=low, green=high confidence


def plot_confidence_site_panel(sites, confidence_path, pred_unfiltered_path,
                                pred_filtered_path,
                                output_name='figure2_confidence_sites'):
    """
    3-row x 3-column panel showing confidence layer behaviour at three sites.
    Columns: confidence raster | all predictions | filtered predictions
    Rows: high / medium / low modelled confidence
    """
    n_sites = len(sites)
    fig, axes = plt.subplots(n_sites, 3, figsize=(12, 4 * n_sites))

    col_titles = [
        'Confidence raster (500 m)',
        'All predictions (unfiltered)',
        f'Filtered (conf ≥ {CONFIG["confidence_threshold"]})',
    ]

    for col, title in enumerate(col_titles):
        axes[0, col].set_title(title, fontweight='bold', pad=6)

    for row, (site_label, bbox) in enumerate(sites.items()):
        w, s, e, n = bbox
        axes[row, 0].set_ylabel(site_label, fontsize=9, labelpad=8)

        for col in range(3):
            ax = axes[row, col]
            ax.set_xlim(w, e)
            ax.set_ylim(s, n)
            ax.set_xticks([])
            ax.set_yticks([])

            if col == 0:
                # Confidence raster
                if os.path.exists(confidence_path):
                    with rasterio.open(confidence_path) as src:
                        window = rasterio.windows.from_bounds(w, s, e, n, src.transform)
                        conf_clip = src.read(1, window=window)
                    im = ax.imshow(conf_clip, cmap=CONF_CMAP, vmin=0, vmax=1,
                                  extent=[w, e, s, n], origin='upper')
                    plt.colorbar(im, ax=ax, shrink=0.8, label='Confidence')
                else:
                    ax.set_facecolor('#dddddd')
                    ax.text(0.5, 0.5, 'Pending', ha='center', va='center',
                           transform=ax.transAxes, color='gray', style='italic')

            elif col == 1:
                # Unfiltered predictions
                path = pred_unfiltered_path
                if os.path.exists(path):
                    with rasterio.open(path) as src:
                        window = rasterio.windows.from_bounds(w, s, e, n, src.transform)
                        pred_clip = src.read(1, window=window).astype(float)
                    ax.imshow(pred_clip, cmap=SETTLEMENT_CMAP, norm=SETTLEMENT_NORM,
                             extent=[w, e, s, n], origin='upper', interpolation='none')
                else:
                    ax.set_facecolor('#dddddd')
                    ax.text(0.5, 0.5, 'Pending', ha='center', va='center',
                           transform=ax.transAxes, color='gray', style='italic')

            else:
                # Filtered predictions
                path = pred_filtered_path
                if os.path.exists(path):
                    with rasterio.open(path) as src:
                        window = rasterio.windows.from_bounds(w, s, e, n, src.transform)
                        pred_clip = src.read(1, window=window).astype(float)
                    n_kept  = (pred_clip > 0).sum()
                    n_total = (pred_clip >= 0).sum()
                    ax.imshow(pred_clip, cmap=SETTLEMENT_CMAP, norm=SETTLEMENT_NORM,
                             extent=[w, e, s, n], origin='upper', interpolation='none')
                    ax.set_xlabel(f'{n_kept:,} / {n_total:,} px kept', fontsize=7)
                else:
                    ax.set_facecolor('#dddddd')
                    ax.text(0.5, 0.5, 'Pending', ha='center', va='center',
                           transform=ax.transAxes, color='gray', style='italic')

    plt.suptitle('Figure 2 — Confidence layer behaviour at representative sites',
                fontweight='bold', y=1.01)
    plt.tight_layout()

    for fmt in ['png', 'pdf']:
        fig.savefig(f'{FIGURE_DIR}/{output_name}.{fmt}')
    plt.show()
    print('Figure 2 saved.')


plot_confidence_site_panel(
    sites=CONFIDENCE_SITES,
    confidence_path='../outputs/confidence_raster_5country.tif',
    pred_unfiltered_path='../outputs/settlement_unfiltered_5country.tif',
    pred_filtered_path='../outputs/settlement_filtered_conf04_5country.tif'
)

## Figure 3: Per-Country Population Bias — Grouped Bar Chart

One bar per dataset per country, analogous to Figure 8 in Láng-Ritter et al. (2025).

In [ ]:
def plot_bias_barchart(bias_csv_path, output_name='figure3_population_bias'):
    """
    Grouped horizontal bar chart of population bias percentages.
    One group per country, three bars per group (WorldPop, GHS-POP, GWP).
    Reference line at 0 (no bias). Negative = underestimation.
    """
    if not os.path.exists(bias_csv_path):
        print(f'Bias CSV not found at {bias_csv_path} — run Notebook 04 first.')
        return

    df = pd.read_csv(bias_csv_path, index_col=0)
    countries = df.index.tolist()
    datasets  = df.columns.tolist()

    dataset_colors = {
        'worldpop': '#0072B2',
        'ghspop':   '#D55E00',
        'gwp':      '#009E73',
    }
    dataset_labels = {
        'worldpop': 'WorldPop',
        'ghspop':   'GHS-POP',
        'gwp':      'GWP v4',
    }

    n_countries = len(countries)
    n_datasets  = len(datasets)
    bar_height  = 0.22
    offsets     = np.linspace(-(n_datasets-1)/2, (n_datasets-1)/2, n_datasets) * bar_height

    fig, ax = plt.subplots(figsize=(9, 4 + 0.5 * n_countries))

    y_positions = np.arange(n_countries)

    for i, dataset in enumerate(datasets):
        values = df[dataset].values
        ax.barh(
            y_positions + offsets[i],
            values,
            height=bar_height,
            color=dataset_colors.get(dataset, 'gray'),
            label=dataset_labels.get(dataset, dataset),
            alpha=0.85
        )

    ax.axvline(0, color='black', linewidth=0.8, linestyle='--')
    ax.set_yticks(y_positions)
    ax.set_yticklabels(countries)
    ax.set_xlabel('Bias percentage (%) — negative = underestimation')
    ax.set_title('Population Bias of Gridded Datasets vs. Settlement-Implied Estimates\n'
                'East African Great Lakes Region', fontweight='bold')
    ax.legend(title='Dataset', loc='lower right')
    ax.invert_yaxis()  # top country first

    # Annotate mean bias per dataset
    for dataset in datasets:
        mean_bias = df[dataset].mean()
        print(f'{dataset_labels.get(dataset, dataset)}: mean bias = {mean_bias:+.1f}%')

    plt.tight_layout()
    for fmt in ['png', 'pdf']:
        fig.savefig(f'{FIGURE_DIR}/{output_name}.{fmt}')
    plt.show()
    print('Figure 3 saved.')


plot_bias_barchart('../outputs/population_bias_results.csv')

## Figure 4: Class-Stratified Bias Decomposition

Novel contribution — bias for isolated-dwelling-dominated vs clustered-settlement-dominated clusters.

In [ ]:
def plot_stratified_bias(stratified_csv_path,
                          output_name='figure4_stratified_bias'):
    """
    Side-by-side bar chart comparing bias for isolated-dominated
    vs clustered-dominated DHS clusters, per country and per dataset.
    """
    if not os.path.exists(stratified_csv_path):
        print(f'Stratified bias CSV not found — run Notebook 04 Sections 4.8 first.')
        return

    df = pd.read_csv(stratified_csv_path, index_col=0)

    fig, axes = plt.subplots(1, 2, figsize=(12, 5), sharey=True)

    titles = ['Isolated-dwelling-dominated clusters', 'Clustered-settlement-dominated clusters']
    cols   = ['isolated_dominated', 'clustered_dominated']

    for ax, col, title in zip(axes, cols, titles):
        if col in df.columns:
            countries = df.index.tolist()
            values    = df[col].values
            colors    = ['#CC0000' if v < 0 else '#009E73' for v in values]

            ax.barh(countries, values, color=colors, alpha=0.85)
            ax.axvline(0, color='black', linewidth=0.8, linestyle='--')
            ax.set_title(title, fontweight='bold')
            ax.set_xlabel('Bias percentage (%)')
            ax.invert_yaxis()
        else:
            ax.text(0.5, 0.5, f'Column "{col}" not found\nin CSV',
                   ha='center', va='center', transform=ax.transAxes,
                   color='gray', style='italic')

    plt.suptitle('Figure 4 — Population bias decomposed by settlement class\n'
                'WorldPop vs. settlement-implied population',
                fontweight='bold')
    plt.tight_layout()

    for fmt in ['png', 'pdf']:
        fig.savefig(f'{FIGURE_DIR}/{output_name}.{fmt}')
    plt.show()
    print('Figure 4 saved.')


plot_stratified_bias('../outputs/bias_stratified_by_class.csv')

## Figure 5: Confidence Model Performance — LOCO Bar Chart

Per-country leave-one-country-out AUC, analogous to Figure 3 (left panel) in Robinson et al. (2026).

In [ ]:
def plot_loco_auc(loco_aucs_dict, cv_mean_auc,
                   output_name='figure5_loco_auc'):
    """
    Horizontal bar chart of LOCO AUC per country with mean line.

    Args:
        loco_aucs_dict: {country: auc} from leave_one_country_out_eval()
        cv_mean_auc: float, 5-fold CV mean AUC for annotation
    """
    if not loco_aucs_dict:
        print('No LOCO results available — run Notebook 04 Section 4.3 first.')
        return

    countries = list(loco_aucs_dict.keys())
    aucs      = [loco_aucs_dict[c] for c in countries]
    mean_auc  = np.mean(aucs)

    # Sort by AUC descending
    sorted_pairs = sorted(zip(aucs, countries), reverse=True)
    aucs_s, countries_s = zip(*sorted_pairs)

    colors = ['#0072B2' if a >= mean_auc else '#E69F00' for a in aucs_s]

    fig, ax = plt.subplots(figsize=(7, 4))
    ax.barh(countries_s, aucs_s, color=colors, alpha=0.85)
    ax.axvline(mean_auc, color='black', linewidth=1.0, linestyle='--',
               label=f'Mean AUC = {mean_auc:.3f}')

    # Annotate values
    for i, (auc, country) in enumerate(zip(aucs_s, countries_s)):
        ax.text(auc + 0.005, i, f'{auc:.3f}', va='center', fontsize=8)

    ax.set_xlim(0.5, 1.0)
    ax.set_xlabel('AUC (leave-one-country-out)')
    ax.set_title('Figure 5 — Confidence model transferability\n'
                'Leave-one-country-out cross-validation (Kenya/Uganda only)',
                fontweight='bold')
    ax.legend()

    # Add 5-fold CV mean for reference
    ax.text(0.98, 0.05, f'5-fold CV AUC = {cv_mean_auc:.3f}',
           ha='right', va='bottom', transform=ax.transAxes,
           fontsize=8, color='gray')

    plt.tight_layout()
    for fmt in ['png', 'pdf']:
        fig.savefig(f'{FIGURE_DIR}/{output_name}.{fmt}')
    plt.show()
    print('Figure 5 saved.')


# Placeholder call — replace with actual LOCO results from Notebook 04
# loco_aucs = {'Kenya': 0.84, 'Uganda': 0.81}  # example
# plot_loco_auc(loco_aucs, cv_mean_auc=0.82)
print('plot_loco_auc() defined — call with results from Notebook 04 Section 4.3.')

## Figure 6: Cumulative Retention vs Confidence Threshold

Shows sensitivity of filtered product to threshold choice — analogous to Figure M1 in Robinson et al. (2026).

In [ ]:
def plot_cumulative_retention(prediction_path, confidence_path,
                               output_name='figure6_cumulative_retention'):
    """
    Two-panel figure:
      Left:  number of settled pixels retained vs confidence threshold
      Right: total implied population retained vs confidence threshold
    """
    if not (os.path.exists(prediction_path) and os.path.exists(confidence_path)):
        print('Prediction or confidence raster not found — run Notebooks 01-04 first.')
        return

    thresholds = np.arange(0.0, 0.75, 0.05)
    n_retained = []

    with rasterio.open(prediction_path) as src_pred, \
         rasterio.open(confidence_path) as src_conf:

        pred = src_pred.read(1)
        conf = src_conf.read(1)

        # Upsample confidence to 10m
        conf_up = np.repeat(np.repeat(conf, 50, axis=0), 50, axis=1)
        conf_up = conf_up[:pred.shape[0], :pred.shape[1]]

        total_settled = (pred > 0).sum()

        for t in thresholds:
            kept = ((pred > 0) & (conf_up >= t)).sum()
            n_retained.append(kept)

    n_retained = np.array(n_retained)
    pct_retained = 100 * n_retained / total_settled

    fig, axes = plt.subplots(1, 2, figsize=(10, 4))

    axes[0].plot(thresholds, n_retained / 1e6, color='#0072B2', linewidth=2)
    axes[0].axvline(CONFIG['confidence_threshold'], color='black',
                   linestyle='--', linewidth=0.8, label=f'Default threshold ({CONFIG["confidence_threshold"]})')
    axes[0].set_xlabel('Confidence threshold')
    axes[0].set_ylabel('Settled pixels retained (millions)')
    axes[0].set_title('Pixel count retention', fontweight='bold')
    axes[0].legend()

    axes[1].plot(thresholds, pct_retained, color='#D55E00', linewidth=2)
    axes[1].axvline(CONFIG['confidence_threshold'], color='black',
                   linestyle='--', linewidth=0.8)
    axes[1].set_xlabel('Confidence threshold')
    axes[1].set_ylabel('Percentage of total settled pixels retained (%)')
    axes[1].set_title('Percentage retention', fontweight='bold')

    plt.suptitle('Figure 6 — Cumulative retention as a function of confidence threshold',
                fontweight='bold')
    plt.tight_layout()

    for fmt in ['png', 'pdf']:
        fig.savefig(f'{FIGURE_DIR}/{output_name}.{fmt}')
    plt.show()
    print('Figure 6 saved.')


plot_cumulative_retention(
    '../outputs/settlement_unfiltered_5country.tif',
    '../outputs/confidence_raster_5country.tif'
)

## Figure Summary

In [ ]:
figures = [
    ('figure1_regional_map',        'Regional settlement detection map with insets'),
    ('figure2_confidence_sites',    'Confidence layer at three representative sites'),
    ('figure3_population_bias',     'Per-country population bias by dataset'),
    ('figure4_stratified_bias',     'Class-stratified bias decomposition (novel)'),
    ('figure5_loco_auc',            'Confidence model LOCO transferability'),
    ('figure6_cumulative_retention','Cumulative retention vs confidence threshold'),
]

print('Figure inventory:')
print()
for name, description in figures:
    for fmt in ['png', 'pdf']:
        path = f'{FIGURE_DIR}/{name}.{fmt}'
        exists = os.path.exists(path)
        status = 'READY' if exists else 'PENDING'
        print(f'  [{status}] {name}.{fmt}')
    print(f'          {description}')
    print()

---
## Pipeline Complete

All five notebooks have been executed. Deliverables produced:

**Geospatial data products** (`../outputs/`)
- `settlement_unfiltered_5country.tif` — full detection map (10m)
- `settlement_filtered_conf04_5country.tif` — default filtered map (conf ≥ 0.4)
- `settlement_filtered_conf05_5country.tif` — conservative filtered map (conf ≥ 0.5)
- `confidence_raster_5country.tif` — continuous confidence surface (500m)

**Results tables** (`../outputs/`)
- `population_bias_results.csv` — bias by country and dataset
- `bias_stratified_by_class.csv` — class-stratified bias decomposition

**Publication figures** (`../outputs/figures/`)
- Figures 1–6 in PNG (300 DPI) and PDF